## Qdrant

In [ ]:
# docker run -p 6333:6333 -p 6334:6334 \
#    -v "$(pwd)/qdrant_storage:/qdrant/storage:z" \
#    qdrant/qdrant

In [2]:
# intiialize Qdrant client
from qdrant_client import QdrantClient, models
client = QdrantClient("http://localhost:6333")

In [2]:
# toy dataset
import json

with open('combined.json', 'r') as f:
    docs = json.load(f)

docs[-50]

{'text': 'Use the following line instead in mounting the current volume to docker for Q4:\n`-v "/${PWD}/ollama_files:/root/.ollama"`',
 'section': 'Module 2: Open-Source LLMs',
 'question': 'Docker: Error: Docker mounted volume adds ;C to end of windows path',
 'course': 'llm-zoomcamp'}

In [3]:
from fastembed import TextEmbedding
TextEmbedding.list_supported_models()

[{'model': 'BAAI/bge-base-en',
  'sources': {'hf': 'Qdrant/fast-bge-base-en',
   'url': 'https://storage.googleapis.com/qdrant-fastembed/fast-bge-base-en.tar.gz',
   '_deprecated_tar_struct': True},
  'model_file': 'model_optimized.onnx',
  'description': 'Text embeddings, Unimodal (text), English, 512 input tokens truncation, Prefixes for queries/documents: necessary, 2023 year.',
  'license': 'mit',
  'size_in_GB': 0.42,
  'additional_files': [],
  'dim': 768,
  'tasks': {}},
 {'model': 'BAAI/bge-base-en-v1.5',
  'sources': {'hf': 'qdrant/bge-base-en-v1.5-onnx-q',
   'url': 'https://storage.googleapis.com/qdrant-fastembed/fast-bge-base-en-v1.5.tar.gz',
   '_deprecated_tar_struct': True},
  'model_file': 'model_optimized.onnx',
  'description': 'Text embeddings, Unimodal (text), English, 512 input tokens truncation, Prefixes for queries/documents: not so necessary, 2023 year.',
  'license': 'mit',
  'size_in_GB': 0.21,
  'additional_files': [],
  'dim': 768,
  'tasks': {}},
 {'model':

In [4]:
EMBEDDING_DIMENSIONALITY = 512

for model in TextEmbedding.list_supported_models():
    if model["dim"] == EMBEDDING_DIMENSIONALITY:
        print(json.dumps(model, indent=2))

{
  "model": "BAAI/bge-small-zh-v1.5",
  "sources": {
    "hf": "Qdrant/bge-small-zh-v1.5",
    "url": "https://storage.googleapis.com/qdrant-fastembed/fast-bge-small-zh-v1.5.tar.gz",
    "_deprecated_tar_struct": true
  },
  "model_file": "model_optimized.onnx",
  "description": "Text embeddings, Unimodal (text), Chinese, 512 input tokens truncation, Prefixes for queries/documents: not so necessary, 2023 year.",
  "license": "mit",
  "size_in_GB": 0.09,
  "additional_files": [],
  "dim": 512,
  "tasks": {}
}
{
  "model": "Qdrant/clip-ViT-B-32-text",
  "sources": {
    "hf": "Qdrant/clip-ViT-B-32-text",
    "url": null,
    "_deprecated_tar_struct": false
  },
  "model_file": "model.onnx",
  "description": "Text embeddings, Multimodal (text&image), English, 77 input tokens truncation, Prefixes for queries/documents: not necessary, 2021 year",
  "license": "mit",
  "size_in_GB": 0.25,
  "additional_files": [],
  "dim": 512,
  "tasks": {}
}
{
  "model": "jinaai/jina-embeddings-v2-small-e

In [5]:
# when selecting embedding model, we have to check dimensionality
# how they are evaluated (cosine similarity vs euclidean distance)
model_handle = "jinaai/jina-embeddings-v2-small-en"

In [42]:
# configure the collection of our dataset
collection_name = "llm-zoomcamp"

client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=EMBEDDING_DIMENSIONALITY, 
        distance=models.Distance.COSINE 
    )
)

True

In [ ]:
# delete collection if needed
# client.delete_collection(collection_name=collection_name)

True

In [43]:
# vectorise each document and prepare points for upload
# "text" + "question" field are converted to embeddings 
# whole document is saved as payloads (metadata)

points = []

for id, doc in enumerate(docs):
    to_vec = doc['text'] + " " + doc['question']
    vector = models.Document(text=to_vec, model=model_handle)

    point = models.PointStruct(
        id=id,
        vector=vector,
        payload=doc
    )
    points.append(point)

In [44]:
# build vector index
client.upsert(
    collection_name=collection_name,
    points=points
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

### Vector Search

In [45]:
# use qdrant to search for similar documents
def search(query, limit=1):
    results = client.query_points(
        collection_name=collection_name,
        query=models.Document(
            text=query,
            model=model_handle 
        ),
        limit=limit, 
        with_payload=True 
    )

    return results

In [46]:
# get random doc question
docs[20]

{'text': 'You can set it up on your laptop or PC if you prefer to work locally from your laptop or PC.\nYou might face some challenges, especially for Windows users. If you face cnd2\nIf you prefer to work on the local machine, you may start with the week 1 Introduction to Docker and follow through.\nHowever, if you prefer to set up a virtual machine, you may start with these first:\nUsing GitHub Codespaces\nSetting up the environment on a cloudV Mcodespace\nI decided to work on a virtual machine because I have different laptops & PCs for my home & office, so I can work on this boot camp virtually anywhere.',
 'section': 'General course-related questions',
 'question': 'Environment - Should I use my local machine, GCP, or GitHub Codespaces for my environment?',
 'course': 'data-engineering-zoomcamp'}

In [48]:
# example search
res = search("Should I use my local machine, GCP, or GitHub Codespaces for my environment?", limit=3)
res.points

[ScoredPoint(id=20, version=1, score=0.88973486, payload={'text': 'You can set it up on your laptop or PC if you prefer to work locally from your laptop or PC.\nYou might face some challenges, especially for Windows users. If you face cnd2\nIf you prefer to work on the local machine, you may start with the week 1 Introduction to Docker and follow through.\nHowever, if you prefer to set up a virtual machine, you may start with these first:\nUsing GitHub Codespaces\nSetting up the environment on a cloudV Mcodespace\nI decided to work on a virtual machine because I have different laptops & PCs for my home & office, so I can work on this boot camp virtually anywhere.', 'section': 'General course-related questions', 'question': 'Environment - Should I use my local machine, GCP, or GitHub Codespaces for my environment?', 'course': 'data-engineering-zoomcamp'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=22, version=1, score=0.8895242, payload={'text': "It's up to you whic

### Filtering

In [50]:
client.create_payload_index(
    collection_name=collection_name,
    field_name="course",
    field_schema="keyword" # exact matching on string metadata fields
)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

In [23]:
# search within a specific course
def search_in_course(query, course="mlops-zoomcamp", limit=1):
    results = client.query_points(
        collection_name=collection_name,
        query=models.Document(
            text=query,
            model=model_handle
        ),
        query_filter=models.Filter( 
            must=[
                models.FieldCondition(
                    key="course",
                    match=models.MatchValue(value=course)
                )
            ]
        ),
        limit=limit, 
        with_payload=True
    )

    return results

In [39]:
# example search
query = "can i still enroll if the course has already started?"

res = search_in_course(query, "machine-learning-zoomcamp", 3)
res.points

[ScoredPoint(id=450, version=1, score=0.85373706, payload={'text': 'The course is available in the self-paced mode too, so you can go through the materials at any time. But if you want to do it as a cohort with other students, the next iterations will happen in September 2023, September 2024 (and potentially other Septembers as well).', 'section': 'General course-related questions', 'course': 'machine-learning-zoomcamp'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=451, version=1, score=0.8175851, payload={'text': 'No, it’s not possible. The form is closed after the due date. But don’t worry, homework is not mandatory for finishing the course.', 'section': 'General course-related questions', 'course': 'machine-learning-zoomcamp'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=449, version=1, score=0.8078607, payload={'text': 'Yes, you can. You won’t be able to submit some of the homeworks, but you can still take part in the course.\nIn order to get

In [40]:
res = search(query, 3)
res.points

[ScoredPoint(id=450, version=1, score=0.85373706, payload={'text': 'The course is available in the self-paced mode too, so you can go through the materials at any time. But if you want to do it as a cohort with other students, the next iterations will happen in September 2023, September 2024 (and potentially other Septembers as well).', 'section': 'General course-related questions', 'course': 'machine-learning-zoomcamp'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=958, version=1, score=0.8451529, payload={'text': 'This course is being offered for the first time, and things will keep changing until a given module is ready, at which point it shall be announced. Working on the material/homework in advance will be at your own risk, as the final version could be different.', 'section': 'General course-related questions', 'course': 'llm-zoomcamp'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=7, version=1, score=0.82041943, payload={'text': 'Yes, we wi

### BM25

In [1]:
client.get_collections()

NameError: name 'client' is not defined

In [5]:
# create collection for BM25
collection_name = "bm25"

client.create_collection(
    collection_name=collection_name,
    sparse_vectors_config={
        "bm25":models.SparseVectorParams(
            modifier=models.Modifier.IDF
        )
    }
)

True

In [7]:
# create point and upsert into collection
points = []

for id, doc in enumerate(docs):
    to_vec = doc['text'] + " " + doc['question']

    point = models.PointStruct(
        id=id,
        vector={
            "bm25": models.Document(
                text=to_vec,
                model="Qdrant/bm25"
            )
        },
        payload=doc
    )
    points.append(point)

# build index
client.upsert(
    collection_name=collection_name,
    points=points
)

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [15]:
# bm25 search function
def search(query, limit=3):
    results = client.query_points(
        collection_name=collection_name,
        query=models.Document(
            text=query,
            model="Qdrant/bm25"
        ),
        using="bm25",
        limit=limit, 
        with_payload=True 
    )

    return results

In [20]:
res = search("OpenSource")
res.points

[ScoredPoint(id=975, version=1, score=8.77572, payload={'text': 'Yes. See module 2 and the open-ai-alternatives.md in module 1 folder.', 'section': 'Module 1: Introduction', 'question': 'OpenSource: Can I use open-source alternatives to OpenAI API?', 'course': 'llm-zoomcamp'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=974, version=1, score=8.555101, payload={'text': 'You can use any LLM platform for your experiments and your project. Also, the homework is designed in such a way that you don’t need to have access to any paid services and can do it locally. However, you would need to adjust the code for that platform. See their documentation pages.', 'section': 'Module 1: Introduction', 'question': 'OpenSource: Can I use Groq instead of OpenAI?', 'course': 'llm-zoomcamp'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=973, version=1, score=8.224292, payload={'text': "The question asks for the number of tokens in gpt-4o model. tiktoken is a python l